# GAN Urban Design v2 — External Modeller (CycleGAN + Pix2PixHD)

**Ön koşul:** `colab_pix2pix.ipynb` notebook'unun Hücre 1–6'sı bu çalışma alanında daha önce çalıştırılmış olmalı (Drive bağlama + repo klonu + `maps` indirme + sentetik sketch üretimi). Bu notebook o adımları tekrar yapar ama yapılmışsa hızlıca atlar.

**Bug fix notu (v2):** Önceki sürümde `%cd /content/{REPO}/...` IPython magic'i, `{REPO}` brace expansion'ını desteklemediği için literal string olarak alıyordu. Bu sürümde tüm dizin değişiklikleri `os.chdir(f'...')` Python f-string'i ile yapılır — %100 güvenilir.

## 1) Drive bağla + repo'yu güncelle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, pathlib, sys
GITHUB_USER = 'ilker-23'
REPO = 'gan-urban-design-v2'
REPO_DIR = f'/content/{REPO}'

os.chdir('/content')
# eski başarısız klonların bıraktığı kalıntıları temizle
if os.path.isdir('github.com'):
    shutil.rmtree('github.com')
if os.path.isdir(REPO) and not os.path.isdir(f'{REPO}/.git'):
    shutil.rmtree(REPO)

if not os.path.isdir(REPO):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git
else:
    !git -C {REPO} pull --ff-only

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
!ls

## 2) Bağımlılıklar

In [ ]:
!pip install -q -r requirements.txt
# CycleGAN/Pix2PixHD'nin ekstra bağımlılıkları
!pip install -q dominate visdom
!nvidia-smi | head -10

## 3) Maps dataset + sentetik kroki (yoksa üret)
Bu hücre, `colab_pix2pix.ipynb`'in 5-6. hücrelerini güvenli biçimde tekrarlar — zaten varsa hızlıca atlar.

In [ ]:
os.chdir(REPO_DIR)

if not os.path.isdir('data/maps/train'):
    print('>> maps dataset indiriliyor...')
    !bash scripts/download_data.sh maps
else:
    print('   data/maps zaten var, atlandı.')

if not os.path.isdir('data/processed/maps_sketch/train/A'):
    print('>> sentetik kroki üretiliyor (~3-5 dk)...')
    !python scripts/prepare_sketches.py --input data/maps --output data/processed --method canny_xdog
else:
    print('   data/processed/maps_sketch zaten var, atlandı.')

# Sağlama
n_train = len(os.listdir('data/processed/maps_sketch/train/A'))
n_val   = len(os.listdir('data/processed/maps_sketch/val/A'))
print(f'\nTrain: {n_train} | Val: {n_val}   (beklenen: 1096 | 1098)')
assert n_train > 1000 and n_val > 1000, 'Veri eksik; öncesi başarısız olmuş olmalı.'

## 4) External repoları çek (CycleGAN + Pix2PixHD)

In [ ]:
os.chdir(REPO_DIR)
!bash scripts/setup_external.sh

# Sağlama
for sub in ('pytorch-CycleGAN-and-pix2pix', 'pix2pixHD'):
    p = f'external/{sub}'
    assert os.path.isdir(p) and os.path.isfile(f'{p}/train.py'), \
        f'EKSİK: {p}/train.py — setup_external.sh başarısız oldu, internet bağlantısı veya git erişimini kontrol edin.'
    print(f'OK  {p}/train.py')
print('\nexternal/ içeriği:')
!ls external/

## 5) Veriyi external formatlara uyumlulaştır (symlink, ek disk yok)

In [ ]:
os.chdir(REPO_DIR)
!python scripts/prepare_external_data.py --src data/processed/maps_sketch --out data/external

# Sağlama
for path, exp in [
    ('data/external/junyanz_cyc/trainA', 1000),
    ('data/external/junyanz_cyc/trainB', 1000),
    ('data/external/junyanz_cyc/testA',  1000),
    ('data/external/junyanz_cyc/testB',  1000),
    ('data/external/pix2pixhd/train_A',  1000),
    ('data/external/pix2pixhd/train_B',  1000),
    ('data/external/pix2pixhd/test_A',   1000),
    ('data/external/pix2pixhd/test_B',   1000),
]:
    n = len(os.listdir(path))
    print(f'{path:45s} {n:>5d} (>= {exp})')
    assert n >= exp, f'{path} eksik içerik'

---
## 6) CycleGAN eğitimi (~2 saat A100)

256×256, 200 epoch (100 sabit + 100 decay). Checkpoint'ler `external/.../checkpoints/sketch2map_cyc/` altına yazılır.

In [ ]:
CYC_DIR = f'{REPO_DIR}/external/pytorch-CycleGAN-and-pix2pix'
assert os.path.isfile(f'{CYC_DIR}/train.py'), 'Hücre 4 başarısız olmuş; setup_external tekrar çalıştır.'
os.chdir(CYC_DIR)
print('cwd:', os.getcwd())

DATA = f'{REPO_DIR}/data/external/junyanz_cyc'
# Not: --display_id argümanı junyanz repo'sunun güncel sürümünde kaldırıldı.
# Visdom server Colab'da olmadığı için connect-fail warning'i çıkar ama eğitim durmaz.
# --display_freq ve --update_html_freq'i çok yüksek tutarak görsel/HTML çıktısını susturuyoruz.
!python train.py \
    --dataroot {DATA} \
    --name sketch2map_cyc \
    --model cycle_gan \
    --load_size 286 --crop_size 256 \
    --batch_size 4 \
    --n_epochs 100 --n_epochs_decay 100 \
    --save_epoch_freq 25 \
    --display_freq 999999 --update_html_freq 999999 --no_html

os.chdir(REPO_DIR)
print('Eğitim bitti; cwd repo köküne döndü.')

## 7) CycleGAN testi (val'da 1098 örneği çevir)

In [ ]:
CYC_DIR = f'{REPO_DIR}/external/pytorch-CycleGAN-and-pix2pix'
os.chdir(CYC_DIR)
DATA = f'{REPO_DIR}/data/external/junyanz_cyc'
!python test.py \
    --dataroot {DATA} \
    --name sketch2map_cyc \
    --model cycle_gan \
    --load_size 256 --crop_size 256 \
    --num_test 1098 \
    --no_dropout
os.chdir(REPO_DIR)
!ls external/pytorch-CycleGAN-and-pix2pix/results/sketch2map_cyc/test_latest/images | head -5
!ls external/pytorch-CycleGAN-and-pix2pix/results/sketch2map_cyc/test_latest/images | wc -l

## 8) CycleGAN değerlendirmesi (FID + SSIM + PSNR + LPIPS + L1)

In [ ]:
os.chdir(REPO_DIR)
!python scripts/evaluate_external.py \
    --results-dir external/pytorch-CycleGAN-and-pix2pix/results/sketch2map_cyc/test_latest/images \
    --pattern junyanz \
    --tag cyclegan_sketch2map

## 9) CycleGAN Drive yedeği

In [ ]:
os.chdir(REPO_DIR)
DST = '/content/drive/MyDrive/thesis_runs/cyclegan_sketch2map'
os.makedirs(DST, exist_ok=True)
!cp -r external/pytorch-CycleGAN-and-pix2pix/checkpoints/sketch2map_cyc {DST}/checkpoints
!cp -r results/external_eval/cyclegan_sketch2map {DST}/eval
!ls {DST}/

---
## 10) Pix2PixHD eğitimi (~4-5 saat A100)

512×512, 150 epoch (100 sabit + 50 decay). `--label_nc 0 --no_instance` ile görüntüden-görüntüye modu açılır.

In [ ]:
HD_DIR = f'{REPO_DIR}/external/pix2pixHD'
assert os.path.isfile(f'{HD_DIR}/train.py'), 'Hücre 4 başarısız olmuş.'
os.chdir(HD_DIR)
print('cwd:', os.getcwd())

DATA = f'{REPO_DIR}/data/external/pix2pixhd'
# Not: --display_id Pix2PixHD master sürümünde yok. --display_freq yüksek + --no_html ile
# görsel/HTML çıktısını susturuyoruz; visdom bağlantı uyarıları zararsız.
!python train.py \
    --name sketch2map_hd \
    --dataroot {DATA} \
    --label_nc 0 --no_instance \
    --resize_or_crop scale_width \
    --loadSize 512 --fineSize 512 \
    --batchSize 2 \
    --niter 100 --niter_decay 50 \
    --save_epoch_freq 25 \
    --display_freq 999999 --no_html

os.chdir(REPO_DIR)

## 11) Pix2PixHD testi

In [ ]:
HD_DIR = f'{REPO_DIR}/external/pix2pixHD'
os.chdir(HD_DIR)
DATA = f'{REPO_DIR}/data/external/pix2pixhd'
!python test.py \
    --name sketch2map_hd \
    --dataroot {DATA} \
    --label_nc 0 --no_instance \
    --resize_or_crop scale_width \
    --loadSize 512 --fineSize 512 \
    --how_many 1098
os.chdir(REPO_DIR)
!ls external/pix2pixHD/results/sketch2map_hd/test_latest/images | head -5

## 12) Pix2PixHD değerlendirmesi

In [ ]:
os.chdir(REPO_DIR)
# pix2pixHD test.py gerçek hedefi kaydetmez; gerçek görseller test_B'den alınır (--real-dir).
!python scripts/evaluate_external.py \
    --results-dir external/pix2pixHD/results/sketch2map_hd/test_latest/images \
    --pattern pix2pixhd \
    --real-dir data/external/pix2pixhd/test_B \
    --tag pix2pixhd_sketch2map

## 13) Pix2PixHD Drive yedeği

In [ ]:
os.chdir(REPO_DIR)
DST = '/content/drive/MyDrive/thesis_runs/pix2pixhd_sketch2map'
os.makedirs(DST, exist_ok=True)
!cp -r external/pix2pixHD/checkpoints/sketch2map_hd {DST}/checkpoints
!cp -r results/external_eval/pix2pixhd_sketch2map {DST}/eval

---
## 14) Tüm sonuçları tek tabloda topla

In [ ]:
import json, pathlib, pandas as pd
os.chdir(REPO_DIR)

runs = {
    'Pix2Pix (bizim)': '/content/drive/MyDrive/thesis_runs/pix2pix_sketch2map/eval/metrics.json',
    'CycleGAN':        'results/external_eval/cyclegan_sketch2map/metrics.json',
    'Pix2PixHD':       'results/external_eval/pix2pixhd_sketch2map/metrics.json',
}

rows = []
for name, path in runs.items():
    p = pathlib.Path(path)
    if not p.exists():
        rows.append({'Model': name, 'FID': None, 'SSIM': None, 'PSNR': None, 'LPIPS': None, 'L1': None})
        continue
    s = json.loads(p.read_text())['summary']
    rows.append({
        'Model': name,
        'FID':   round(s.get('fid',         float('nan')), 2),
        'SSIM':  round(s['ssim_mean'],  4),
        'PSNR':  round(s['psnr_mean'],  2),
        'LPIPS': round(s['lpips_mean'], 4),
        'L1':    round(s['l1_mean'],    2),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
out_csv = '/content/drive/MyDrive/thesis_runs/comparison_table.csv'
df.to_csv(out_csv, index=False)
print(f'\nKaydedildi: {out_csv}')

---
## Sorun Giderme

**Q: `train.py: No such file or directory` hatası alıyorum.**
A: Hücre 4 (`setup_external.sh`) başarısız olmuş. Hücre 4'ü tekrar çalıştırın; başarılı olunca 'OK external/pytorch-CycleGAN-and-pix2pix/train.py' satırını görmelisiniz.

**Q: Eğitim bitmeden Colab kopuyor.**
A: Pro+ yoksa 12 saatlik limit. CycleGAN için `--n_epochs 50 --n_epochs_decay 50` (yarı süre) kullanın. Drive yedek otomatik olduğu için checkpoint kaybolmaz.

**Q: `CUDA out of memory` aldım.**
A: CycleGAN için `--batch_size 2`; Pix2PixHD için `--batchSize 1` veya `--fineSize 384` deneyin.

**Q: Hücreyi yeniden çalıştırınca cwd `/content`'te kalıyor.**
A: Bu sürümde `os.chdir(REPO_DIR)` her hücrenin başında yapılır — `%cd` magic'in brace expansion bug'ı aşılmıştır.

---
## 15) ⭐ ÖNERİLEN: Geliştirilmiş Pix2Pix — 512² + VGG perceptual loss

**Pix2PixHD'nin 2018 NVIDIA repo'su Python 3.12 ile uyumsuzluk çıkarabiliyor** (`fractions.gcd` vb.; `scripts/patch_pix2pixhd.py` ile otomatik yamalanır ama eski PyTorch API'leri yüzünden başka hatalar da çıkabilir). 

Bu bölüm, **kendi kod tabanımızla** (`src/train.py`, tam tekrarlanabilir) Pix2PixHD'nin **iki ana katkısını** uygular:
1. **Yüksek çözünürlük** — 256² yerine 512² (U-Net num_downs=9)
2. **VGG perceptual loss** — λ_VGG=10 (baseline'da kapalıydı)

Bu, jüriye karşı savunulması en kolay, %100 çalışan üçüncü modeldir. **Süre:** ~2-3 saat A100.

> Pix2PixHD'nin gerçek (multi-scale discriminator'lü) sürümünü denemek isterseniz Hücre 10-13'ü kullanın; patch otomatik uygulanır ama başarı garanti değildir."

In [ ]:
# Geliştirilmiş Pix2Pix eğitimi: 512² + VGG perceptual loss (kendi src/train.py)
import os, pathlib
RUN_DIR_ENH = '/content/drive/MyDrive/thesis_runs/pix2pix_enhanced'
pathlib.Path(RUN_DIR_ENH).mkdir(parents=True, exist_ok=True)
os.environ['RUN_DIR'] = RUN_DIR_ENH
os.chdir(REPO_DIR)
!python -m src.train --config configs/pix2pix_enhanced.yaml

In [ ]:
# Geliştirilmiş Pix2Pix değerlendirmesi (kendi src/evaluate.py — FID/SSIM/PSNR/LPIPS/L1)
import glob, os
os.chdir(REPO_DIR)
os.environ['RUN_DIR'] = RUN_DIR_ENH
ckpts = sorted(glob.glob(f'{RUN_DIR_ENH}/checkpoints/pix2pix_epoch_*.pth'))
print('Bulunan checkpointler:', ckpts[-3:])
LAST = ckpts[-1]
!python -m src.evaluate --config configs/pix2pix_enhanced.yaml --checkpoint {LAST}
# Sonuçlar RUN_DIR_ENH/eval/metrics.json olarak Drive'a yazılır (otomatik yedek).
!cat {RUN_DIR_ENH}/eval/metrics.json | python -c "import json,sys; print(json.load(sys.stdin)['summary'])"

---
## 16) Nihai 4-model karşılaştırma tablosu
Pix2Pix + CycleGAN + (varsa) Pix2PixHD + Geliştirilmiş Pix2Pix sonuçlarını tek tabloda toplar.

In [ ]:
import json, pathlib, pandas as pd
os.chdir(REPO_DIR)

runs = {
    'Pix2Pix (256, L1)':         '/content/drive/MyDrive/thesis_runs/pix2pix_sketch2map/eval/metrics.json',
    'CycleGAN (256, unpaired)':  'results/external_eval/cyclegan_sketch2map/metrics.json',
    'Pix2PixHD (512)':           'results/external_eval/pix2pixhd_sketch2map/metrics.json',
    'Pix2Pix-Enhanced (512+VGG)':'/content/drive/MyDrive/thesis_runs/pix2pix_enhanced/eval/metrics.json',
}

rows = []
for name, path in runs.items():
    p = pathlib.Path(path)
    if not p.exists():
        rows.append({'Model': name, 'FID': None, 'SSIM': None, 'PSNR': None, 'LPIPS': None, 'L1': None})
        continue
    s = json.loads(p.read_text())['summary']
    rows.append({
        'Model': name,
        'FID':   round(s.get('fid', float('nan')), 2),
        'SSIM':  round(s['ssim_mean'],  4),
        'PSNR':  round(s['psnr_mean'],  2),
        'LPIPS': round(s['lpips_mean'], 4),
        'L1':    round(s['l1_mean'],    2),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
out_csv = '/content/drive/MyDrive/thesis_runs/comparison_table_full.csv'
df.to_csv(out_csv, index=False)
print(f'\nKaydedildi: {out_csv}')